<a href="https://colab.research.google.com/github/Adarsha2004/finetuning/blob/main/finetuning_cpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 9a: Continued Pre-Training (CPT) from Scratch
## SmolLM-135M + SEC 10-K Filings + Unsloth

**What we'll do:**
1. Load SmolLM-135M and try financial text (watch it fail)
2. Download & prepare SEC 10-K filings
3. CPT with Unsloth + LoRA
4. Compare before vs after (generation + perplexity)
5. Experiment: with vs without data mixing (catastrophic forgetting)
6. Tease SFT: ask it a question, see it can't answer

**Requirements:** Google Colab with T4 GPU (free tier works!)

---
## Part 0: Setup

In [ ]:
# Install Unsloth (handles LoRA, quantization, fast training)
# This takes ~2 minutes on Colab
!pip install -q unsloth
!pip install -q datasets evaluate rouge_score

# Suppress noisy deprecation warnings
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*use_return_dict.*')
warnings.filterwarnings('ignore', message='.*max_new_tokens.*max_length.*')

In [ ]:
import torch
import json
import random
import math
import re
import os
from pathlib import Path

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## Part 1: Meet the Base Model

Let's load SmolLM-135M and see what it knows (and doesn't know) about financial language.

In [ ]:
from unsloth import FastLanguageModel

BASE_MODEL = "HuggingFaceTB/SmolLM-135M"
MAX_SEQ_LENGTH = 512
SEED = 42

# Load the base model (no fine-tuning yet)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Set up for inference
FastLanguageModel.for_inference(model)

print(f"\n[OK] Model loaded: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Helper function to generate text
def generate_text(model, tokenizer, prompt, max_new_tokens=100):
    """Generate a completion for a given prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            repetition_penalty=1.2,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    # Only decode the NEW tokens (not the prompt)
    new_tokens = outputs[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:
# Test the base model on financial prompts
financial_prompts = [
    "The company reported total revenue of",
    "Risk factors include exposure to fluctuations in",
    "Diluted earnings per share for the fiscal year ended",
    "The Board of Directors declared a quarterly dividend of",
    "Our long-term debt obligations consist primarily of",
]

print("=" * 70)
print("BASE MODEL -- Financial Text Generation")
print("=" * 70)
for prompt in financial_prompts:
    completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {completion[:200]}")
    print("-" * 50)

**Observation:** The base model generates generic text. It has no idea what financial language sounds like. It might produce grammatical English, but it won't sound like a 10-K filing.

Let's also measure how "surprised" the model is by real financial text.

In [ ]:
def compute_perplexity(model, tokenizer, texts, max_length=512):
    """
    Perplexity = e^(average cross-entropy loss)
    Lower = model is less surprised by the text = better fit

    This is THE metric for CPT. If perplexity drops on domain text
    after training, the model learned the domain.
    """
    total_loss = 0
    total_tokens = 0

    model.eval()
    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs.input_ids)

        num_tokens = inputs.input_ids.shape[1]
        total_loss += outputs.loss.item() * num_tokens
        total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    return math.exp(avg_loss)

In [ ]:
# Quick perplexity check on sample financial text
sample_financial = [
    "The Company recognized impairment charges of $142 million related to goodwill associated with the North America reporting unit during the fiscal year ended March 31, 2024.",
    "Net cash provided by operating activities was $3.8 billion for the year ended December 31, 2023, compared to $4.1 billion for the year ended December 31, 2022.",
    "Total stockholders equity decreased to $12.4 billion as of September 30, 2023, primarily due to share repurchases of $2.1 billion and dividend payments of $800 million.",
]

base_ppl = compute_perplexity(model, tokenizer, sample_financial)
print(f"Base model perplexity on financial text: {base_ppl:.1f}")
print("(Higher = more surprised = doesn't know this domain well)")

---
## Part 2: Prepare the Data

We'll use SEC 10-K filings from HuggingFace.

**10-K** = annual report every public US company files with the SEC. Dense, formal, jargon-heavy financial language -- perfect for CPT.

In [ ]:
from datasets import load_dataset

print("[...] Loading SEC 10-K filings from HuggingFace...")
print("This is the PleIAs/SEC dataset -- full annual reports 1993-2024.")

# Stream to avoid downloading 100GB+
sec_dataset = load_dataset(
    "PleIAs/SEC",
    split="train",
    streaming=True
)

# Collect a subset of filings
NUM_TRAIN = 80
NUM_VAL = 20
all_texts = []

print(f"[...] Collecting {NUM_TRAIN + NUM_VAL} filings...")
for i, example in enumerate(sec_dataset):
    text = example.get("text", "")
    # Filter: only keep substantial filings (>1000 words)
    if text and len(text.split()) > 1000:
        all_texts.append(text)
    if len(all_texts) >= NUM_TRAIN + NUM_VAL:
        break
    if i % 500 == 0 and i > 0:
        print(f"  Scanned {i} entries, collected {len(all_texts)} so far...")

print(f"[OK] Collected {len(all_texts)} filings")

In [ ]:
# Let's look at what a 10-K filing looks like
sample = all_texts[0]
words = sample.split()
print(f"Sample filing length: {len(words)} words")
print(f"\nFirst 200 words:")
print(" ".join(words[:200]))
print("\n...")
print(f"\nLast 100 words:")
print(" ".join(words[-100:]))

### Data Cleaning

SEC filings have boilerplate at the end (exhibits, signatures, legal text). We strip that -- same idea as Avik's reference-stripping for arxiv papers.

In [ ]:
def clean_sec_filing(text):
    """
    Clean a SEC 10-K filing:
    1. Strip trailing exhibits/signatures (noise for CPT)
    2. Remove excessive whitespace
    3. Keep the substantive business/financial content
    """
    end_markers = [
        r"(?i)\bEXHIBIT\s+INDEX\b",
        r"(?i)\bSIGNATURES\b",
        r"(?i)\bPOWER\s+OF\s+ATTORNEY\b",
        r"(?i)\bEXHIBIT\s+\d",
    ]

    earliest_end = len(text)
    for pattern in end_markers:
        matches = list(re.finditer(pattern, text))
        if matches:
            last_match = matches[-1]
            if last_match.start() > 0.7 * len(text):
                earliest_end = min(earliest_end, last_match.start())

    text = text[:earliest_end]
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)
    text = text.strip()
    return text


# Clean all filings
cleaned_texts = []
for i, text in enumerate(all_texts):
    original_len = len(text.split())
    cleaned = clean_sec_filing(text)
    cleaned_len = len(cleaned.split())

    if cleaned_len > 500:
        cleaned_texts.append(cleaned)

    if i < 3:
        removed_pct = (1 - cleaned_len / original_len) * 100
        print(f"Filing {i}: {original_len} -> {cleaned_len} words ({removed_pct:.1f}% removed)")

print(f"\n[OK] {len(cleaned_texts)} filings after cleaning")

### Chunking

SEC filings are very long (10,000+ words). We break them into chunks that fit in the model's context window.

We use word-level chunking with overlap so no information is lost at chunk boundaries.

In [ ]:
def chunk_texts(texts, chunk_size=256, overlap=0.2):
    """
    Split texts into overlapping chunks.

    Why overlap? Without it, a sentence like "Revenue increased by 12%
    compared to the prior year" might get split between "12%" and
    "compared", losing the connection.
    """
    step = int(chunk_size * (1 - overlap))
    all_chunks = []

    for text in texts:
        words = text.split()
        for i in range(0, len(words), step):
            chunk = words[i:i + chunk_size]
            if len(chunk) > 50:  # Skip tiny trailing chunks
                all_chunks.append(" ".join(chunk))

    return all_chunks


# Split into train and val FIRST (at filing level, not chunk level!)
# This prevents data leakage
random.seed(SEED)
random.shuffle(cleaned_texts)

train_filings = cleaned_texts[:NUM_TRAIN]
val_filings = cleaned_texts[NUM_TRAIN:NUM_TRAIN + NUM_VAL]

print(f"Train filings: {len(train_filings)}")
print(f"Val filings:   {len(val_filings)}")

# Now chunk each set
train_chunks = chunk_texts(train_filings, chunk_size=256, overlap=0.2)
val_chunks = chunk_texts(val_filings, chunk_size=256, overlap=0.2)

print(f"\nTrain chunks: {len(train_chunks)}")
print(f"Val chunks:   {len(val_chunks)}")
print(f"\nSample chunk (first 100 words):")
print(" ".join(train_chunks[0].split()[:100]))

In [ ]:
# Convert to HuggingFace Dataset format
from datasets import Dataset

train_dataset = Dataset.from_dict({"text": train_chunks})
val_dataset = Dataset.from_dict({"text": val_chunks})

print(f"Train dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

In [ ]:
# IMPORTANT: Measure base model perplexity on the ACTUAL val chunks
# We need this before we reload the model for training
val_sample = val_chunks[:50]
base_ppl_val = compute_perplexity(model, tokenizer, val_sample)
print(f"Base model perplexity on SEC validation chunks: {base_ppl_val:.1f}")
print("(We'll compare this to the CPT model later)")

---
## Part 3: CPT with Unsloth + LoRA

Now the main event. We'll:
1. Reload the base model (fresh, no inference mode)
2. Add LoRA adapters
3. Train on our SEC filing chunks

Key CPT decisions explained in comments.

In [ ]:
# Free memory from the inference model
del model
torch.cuda.empty_cache()

# Reload fresh for training
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

print(f"[OK] Fresh model loaded for training")

In [ ]:
# Add LoRA adapters
#
# KEY CPT DECISIONS:
# - rank=32: Higher than typical SFT (8-16) because we're teaching
#   new knowledge, not just changing behavior
# - target_modules includes embed_tokens + lm_head
#   This is CPT-specific! SFT usually skips these.
#   We need them because the model is learning new domain vocabulary.
# - lora_alpha=32: Equal to rank, so effective scaling = 1

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",    # Attention
        "gate_proj", "up_proj", "down_proj",          # FFN (where knowledge lives!)
        "embed_tokens", "lm_head",                    # CPT-specific!
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
)

# How many parameters are we actually training?
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"\nTotal parameters:     {total_params:>12,}")
print(f"Trainable (LoRA):     {trainable_params:>12,}  ({100*trainable_params/total_params:.2f}%)")
print(f"Frozen (base model):  {frozen_params:>12,}  ({100*frozen_params/total_params:.2f}%)")

**Look at those numbers!** We're training a tiny fraction of the model's parameters. The rest are frozen -- that's our forgetting protection. The frozen weights preserve general English; the LoRA adapters learn finance.

In [ ]:
from unsloth.trainer import UnslothTrainer, UnslothTrainingArguments

# Training configuration -- every parameter explained:
training_args = UnslothTrainingArguments(
    output_dir="./cpt_sec_filings",

    # --- Core training ---
    num_train_epochs=2,                 # 2 epochs for CPT demo
    per_device_train_batch_size=16,     # Sequences per GPU per step
    gradient_accumulation_steps=2,      # Effective batch = 16 * 2 = 32

    # --- Learning rate ---
    learning_rate=2e-4,                 # Standard for LoRA
    embedding_learning_rate=2e-5,       # 10x LOWER for embeddings!
    # ^ Critical for CPT. embed_tokens affects EVERY token,
    #   so large updates there can destabilize the entire model.
    warmup_steps=50,                    # Gentle ramp-up
    lr_scheduler_type="cosine",         # Decay smoothly to near-zero

    # --- Optimization ---
    optim="adamw_8bit",                 # 8-bit AdamW saves VRAM
    weight_decay=0.01,                  # Light regularization
    max_grad_norm=1.0,                  # Clip gradients to prevent spikes

    # --- Data handling ---
    max_length=MAX_SEQ_LENGTH,
    packing=True,                       # Pack short texts together (no wasted padding!)
    dataset_text_field="text",
    dataset_num_proc=2,

    # --- Evaluation ---
    eval_strategy="steps",
    eval_steps=25,
    per_device_eval_batch_size=16,

    # --- Checkpointing ---
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    # --- Logging ---
    logging_steps=10,
    seed=SEED,
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
)

print("[OK] Trainer configured. Ready to train!")
print(f"Training samples: {len(train_dataset)}")
print(f"Eval samples: {len(val_dataset)}")

In [ ]:
# TRAIN!
# On a T4 GPU with 135M model + 80 filings + 2 epochs, ~8-12 minutes
print("=" * 70)
print("TRAINING STARTED")
print("=" * 70)
print("Watch the eval_loss -- that's your north star.")
print("It should decrease and then plateau.")
print()

trainer_stats = trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

---
## Part 4: Compare Before vs After

The moment of truth. Let's see if our model learned financial language.

In [ ]:
# Switch to inference mode
FastLanguageModel.for_inference(model)

# Generate from the SAME prompts we tried before
print("=" * 70)
print("AFTER CPT -- Financial Text Generation")
print("=" * 70)
for prompt in financial_prompts:
    completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {completion[:200]}")
    print("-" * 50)

In [ ]:
# Perplexity comparison -- the real test
# Compare on the SAME val chunks we measured the base model on

cpt_ppl = compute_perplexity(model, tokenizer, val_sample)
print(f"\nPerplexity on SEC validation chunks:")
print(f"  Base model (before CPT):  {base_ppl_val:.1f}")
print(f"  After CPT:                {cpt_ppl:.1f}")
print(f"  Improvement:              {((base_ppl_val - cpt_ppl) / base_ppl_val * 100):.1f}%")
print()
if cpt_ppl < base_ppl_val:
    print("[PASS] Model is significantly less surprised by financial text!")
else:
    print("[WARN] Perplexity didn't improve -- might need more data or epochs")

---
## Part 5: The Forgetting Experiment

Did CPT make the model forget general English? Let's test on non-financial text.

In [ ]:
# Test on general English text
general_prompts = [
    "The cat sat on the",
    "In a galaxy far far away",
    "The best way to learn programming is to",
    "Once upon a time there was a",
    "The weather today is expected to be",
]

print("=" * 70)
print("AFTER CPT -- General English (Forgetting Check)")
print("=" * 70)
for prompt in general_prompts:
    completion = generate_text(model, tokenizer, prompt, max_new_tokens=60)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {completion[:200]}")
    print("-" * 50)

In [ ]:
# Perplexity on general text
general_texts = [
    "The weather was beautiful that morning as the children walked to school through the park.",
    "Scientists have discovered a new species of butterfly in the Amazon rainforest.",
    "The recipe calls for two cups of flour, one egg, and a tablespoon of butter.",
    "The basketball game went into overtime after a dramatic three-point shot.",
    "She opened the book and began reading the first chapter aloud to her students.",
]

general_ppl_after_cpt = compute_perplexity(model, tokenizer, general_texts)
print(f"Perplexity on general English (after CPT): {general_ppl_after_cpt:.1f}")
print()
print("If this is much higher than expected, the model has 'forgotten'")
print("some general English. This is CATASTROPHIC FORGETTING in action.")
print()
print("LoRA helps prevent this (base weights are frozen), but it's not perfect.")
print("Solution: mix in general data during training.")

---
## Part 6: CPT with Data Mixing (Anti-Forgetting)

The fix: train on 80% domain data + 20% general data.
This gives the optimizer a signal to maintain general capabilities.

In [ ]:
# Reload fresh model for the mixing experiment
del model, trainer
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "embed_tokens", "lm_head",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
)

print("[OK] Fresh model loaded for mixing experiment")

In [ ]:
# Load general text data for mixing
print("[...] Loading general text data (wikitext)...")
wiki_data = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")

# Filter out empty lines and very short text
wiki_texts = [t for t in wiki_data["text"] if t.strip() and len(t.split()) > 50]
random.seed(SEED)
random.shuffle(wiki_texts)

# ~20% of total training data should be general text
num_general_chunks = len(train_chunks) // 4
general_chunks = chunk_texts(wiki_texts[:500], chunk_size=256, overlap=0.2)[:num_general_chunks]

print(f"Domain chunks: {len(train_chunks)}")
print(f"General chunks: {len(general_chunks)}")
total = len(train_chunks) + len(general_chunks)
print(f"Mix ratio: {len(train_chunks)/total*100:.0f}% domain, {len(general_chunks)/total*100:.0f}% general")

In [ ]:
# Combine and shuffle
mixed_train = Dataset.from_dict({
    "text": train_chunks + general_chunks
}).shuffle(seed=SEED)

print(f"Mixed training dataset: {len(mixed_train)} chunks")

In [ ]:
# Train with the mixed data (same config as before)
training_args_mix = UnslothTrainingArguments(
    output_dir="./cpt_sec_mixed",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    embedding_learning_rate=2e-5,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    max_length=MAX_SEQ_LENGTH,
    packing=True,
    dataset_text_field="text",
    dataset_num_proc=2,
    eval_strategy="steps",
    eval_steps=25,
    per_device_eval_batch_size=16,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    seed=SEED,
)

trainer_mix = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=mixed_train,
    eval_dataset=val_dataset,
    args=training_args_mix,
)

print("=" * 70)
print("TRAINING WITH DATA MIXING (80/20)")
print("=" * 70)
trainer_mix_stats = trainer_mix.train()
print(f"\nFinal train loss: {trainer_mix_stats.training_loss:.4f}")

In [ ]:
# Compare: Mixed vs Pure domain training
FastLanguageModel.for_inference(model)

mixed_sec_ppl = compute_perplexity(model, tokenizer, val_sample)
mixed_general_ppl = compute_perplexity(model, tokenizer, general_texts)

print("=" * 70)
print("FORGETTING EXPERIMENT RESULTS")
print("=" * 70)
print()
print(f"{'Metric':<35} {'Base':>10} {'Pure CPT':>10} {'Mixed CPT':>10}")
print("-" * 70)
print(f"{'Perplexity (SEC filings)':.<35} {base_ppl_val:>10.1f} {cpt_ppl:>10.1f} {mixed_sec_ppl:>10.1f}")
print(f"{'Perplexity (General English)':.<35} {'--':>10} {general_ppl_after_cpt:>10.1f} {mixed_general_ppl:>10.1f}")
print()
print("What to look for:")
print("  - Pure CPT: great on SEC, worse on general (forgetting!)")
print("  - Mixed CPT: slightly worse on SEC, but maintains general ability")
print("  - This is the tradeoff. In production, you almost always want mixing.")

---
## Part 7: The SFT Tease

Our model now SOUNDS like a 10-K filing. But can it ANSWER questions about finance?

In [ ]:
print("=" * 70)
print("THE LIMIT OF CPT")
print("=" * 70)
print()

question_prompts = [
    "Question: What was Apple's total revenue in fiscal year 2023?\nAnswer:",
    "Question: What is EBITDA and why do investors care about it?\nAnswer:",
    "Question: Explain the difference between operating income and net income.\nAnswer:",
]

for prompt in question_prompts:
    completion = generate_text(model, tokenizer, prompt, max_new_tokens=80)
    print(f"Prompt: {prompt}")
    print(f"Output: {completion[:250]}")
    print()
    print("-" * 50)
    print()

print("=" * 70)
print("OBSERVATION:")
print("  The model CONTINUES the text instead of ANSWERING the question.")
print("  CPT taught it financial LANGUAGE, not financial BEHAVIOR.")
print()
print("  To make it actually answer questions, we need:")
print("  SUPERVISED FINE-TUNING (SFT) -- next session!")
print("=" * 70)

---
## Part 8: Save the Model

In [ ]:
# Save LoRA adapter
SAVE_PATH = "./cpt_sec_adapter"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

adapter_size = sum(
    os.path.getsize(os.path.join(SAVE_PATH, f))
    for f in os.listdir(SAVE_PATH) if f.endswith(('.safetensors', '.bin'))
)
print(f"[OK] Adapter saved to {SAVE_PATH}")
print(f"Adapter size: {adapter_size / 1e6:.1f} MB")
print(f"Full model would be: ~270 MB")
print(f"Savings: {(1 - adapter_size / 270e6) * 100:.0f}%")

---
## Summary

| Step | What | Why |
|------|------|-----|
| Base model test | Generated financial text with SmolLM | See it fail -- motivation for CPT |
| Data prep | Downloaded SEC 10-K filings, cleaned, chunked | Real-world data pipeline |
| CPT training | LoRA (rank 32) with Unsloth on domain text | Teach the model financial language |
| Before/after | Compared generation quality + perplexity | Prove it worked |
| Forgetting test | Checked general English after CPT | Understand the tradeoff |
| Data mixing | 80% domain + 20% general | Solution to catastrophic forgetting |
| SFT tease | Asked questions -- model can't answer | Motivation for next session |

**Key takeaway:** CPT teaches the model to SOUND like a domain. It does NOT teach the model to be HELPFUL. That's SFT -- next session.

**Next session:** We take this CPT'd model and fine-tune it on financial Q&A pairs (`virattt/financial-qa-10K`) so it actually answers questions about finance.